# Segmenting Retail Customers

**The business question.** Marlow & Finch is a home-goods retailer. Every customer gets the same weekly email and the same discount codes. The marketing director suspects that "customer" is really several different kinds of customer, and that one offer for all of them is wasting money on some and missing others. Are there distinct kinds of customer in the data, and would different offers pay?

**About the data.** Both retail files in this notebook are **synthetic**: Marlow & Finch is fictional, and the customers and orders were generated for teaching. They behave like real retail data, but nobody in them is real. The second part uses a real-style survey of business customers.

**How this notebook works.** There are two parts. Each part follows the same five steps:

1. Load and look
2. Run the model
3. Read the output in plain English
4. Check against JMP
5. Say it to a manager

Copilot is switched on, so you can ask it for code. Check what it gives you against the JMP result, because JMP is your ground truth. There is a Stretch space at the very end for when you are done and want to try something of your own.

Clustering is **unsupervised**: there is no outcome column to predict and no answer key to mark you against. That makes the checking steps matter more, not less.

## Set up

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("Libraries loaded.")

### Settings used throughout this module

These are fixed so that everyone gets the same numbers. Change them only when a step tells you to.

In [ ]:
N_INIT = 25                      # k-means tries 25 different starting points and keeps the best one
SEED = 1                         # fixes the random choices so a re-run gives the same answer
REFERENCE_DATE = "2025-07-01"    # recency is counted in days up to this date (the day after the last order)

---
# Part 1. Demo: segment Marlow & Finch customers

## Step 1. Load and look

Two linked tables: one row per customer, and one row per order. A customer's behaviour is spread across many order rows, so before any clustering we need **one row per customer**.

In [ ]:
customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")

print("customers:", customers.shape, "  orders:", orders.shape)
print("customers who appear in the orders table:", orders["CustomerID"].nunique())
print()
print(customers.head(3).to_string(index=False))
print()
print(orders.head(3).to_string(index=False))

**What to look for.** The customers table has more customers than the orders table has different customer IDs. Some customers never bought anything. Hold that thought, because it matters for what we do with them.

### Building one row per customer

Clustering needs a table with one row per customer and a few columns that describe how they behave. Retailers use three ideas for this: **recency** (how long since the last order), **frequency** (how many orders) and **value** (how much per order). We add a fourth, the share of orders bought with a discount.

The query below does the work in SQL. You are reading it here, not writing it. The SQL notebook gives you full practice.

In [ ]:
query = """
SELECT c.CustomerID,
       c.Region,
       CAST(julianday(:ref) - julianday(MAX(o.OrderDate)) AS INTEGER) AS RecencyDays,
       COUNT(o.OrderID)    AS Orders,
       AVG(o.OrderValue)   AS AvgOrder,
       AVG(o.Discounted)   AS DiscountShare,
       SUM(o.OrderValue)   AS TotalSpend
FROM customers AS c
LEFT JOIN orders AS o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.Region
"""

con = sqlite3.connect(":memory:")                      # a small database that lives in memory
customers.to_sql("customers", con, index=False)        # load both tables into it
orders.to_sql("orders", con, index=False)
features = pd.read_sql(query, con, params={"ref": REFERENCE_DATE})

print(features.shape)
print(features.head().round(2).to_string(index=False))

**Reading the query.**

- `LEFT JOIN` keeps every customer, including those with no orders.
- `GROUP BY c.CustomerID` collapses each customer's orders into one row.
- `MAX(o.OrderDate)` is the last order date, turned into days before the reference date.
- `COUNT`, `AVG` and `SUM` give frequency, average order value, discount share and total spend.

In [ ]:
no_orders = features["RecencyDays"].isna().sum()
print(f"{no_orders} customers have never ordered ({no_orders / len(features):.1%} of the {len(features):,}).")

# A customer who never ordered has no recency, so we set those customers aside and cluster the buyers
buyers = features.dropna(subset=["RecencyDays"]).reset_index(drop=True)
print(f"Customers we will cluster: {len(buyers):,}")
print()
print(buyers[["RecencyDays", "Orders", "AvgOrder", "DiscountShare", "TotalSpend"]].describe().round(2).to_string())

**What to look for.** The customers who never bought have no recency, so they cannot join the clustering. They are a group in their own right ("never bought") and the retailer should hear about them. Note the range of each column.

## Step 2. Run the model

**k-means** picks k points as group centres, assigns every customer to the nearest centre, moves each centre to the middle of its customers, and repeats until nothing moves. You choose k. The data does not.

Two numbers help you choose:

- **Inertia**: how spread out customers are around their own centre. It always falls as k rises, so look for the **elbow**, where the gains flatten.
- **Silhouette**: how well each customer fits its own group compared with the next nearest. Higher is better, and 1 would be perfect.

In [ ]:
FEATURES = ["RecencyDays", "Orders", "AvgOrder", "DiscountShare"]
feature_matrix = buyers[FEATURES].to_numpy()          # the numbers the model measures distance on
print(feature_matrix.shape)

In [ ]:
rows = []
for k in range(2, 9):
    km_k = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(feature_matrix)
    rows.append({"k": k, "inertia": km_k.inertia_,
                 "silhouette": silhouette_score(feature_matrix, km_k.labels_)})
sweep = pd.DataFrame(rows)
print(sweep.round({"inertia": 1, "silhouette": 3}).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sweep["k"], sweep["inertia"], marker="o")
ax[0].set_title("Inertia (look for the elbow)")
ax[1].plot(sweep["k"], sweep["silhouette"], marker="o", color="tab:orange")
ax[1].set_title("Silhouette (higher is better)")
for a in ax:
    a.set_xlabel("k, the number of groups")
plt.tight_layout()
plt.show()

**Choosing k.** The two charts help, but they do not decide. The marketing team can realistically run about four distinct campaigns, so **we use k = 4**. That is also the number every notebook in this module uses for this data, so your numbers can be compared with everyone else's. In real work you would try the neighbours (3 and 5) and keep the one that gives groups you can act on.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEED).fit(feature_matrix)
buyers["Group"] = km.labels_

print("Customers in each group:")
print(buyers["Group"].value_counts().sort_index().to_string())

## Step 3. Read the output in plain English

Group numbers are arbitrary. The model can label the same group 0 in one run and 3 in another. **Read the averages, never the numbers.**

A **profile table** shows what a typical customer in each group looks like. We add two business columns: each group's share of customers and its share of revenue.

In [ ]:
profile = buyers.groupby("Group").agg(
    Customers=("CustomerID", "size"),
    RecencyDays=("RecencyDays", "mean"),
    Orders=("Orders", "mean"),
    AvgOrder=("AvgOrder", "mean"),
    DiscountShare=("DiscountShare", "mean"),
    TotalSpend=("TotalSpend", "mean"),
)
profile["ShareOfCustomers"] = profile["Customers"] / profile["Customers"].sum() * 100
revenue = buyers.groupby("Group")["TotalSpend"].sum()
profile["ShareOfRevenue"] = revenue / revenue.sum() * 100
print(profile.round(2).to_string())

**What to look for.** Compare the rows. Which groups buy often, which buy big, which have gone quiet, which live on discounts? Then compare the last two columns: does a group's share of customers match its share of revenue?

**Name each group.** Give every group a short name that says what kind of customer it is, for example "quiet spenders". Do not use the group numbers.

**Your answer:**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
share = profile[["ShareOfCustomers", "ShareOfRevenue"]]
share.plot(kind="bar", ax=ax, color=["#9aa5a0", "#E07B39"])
ax.set_ylabel("% of buyers or of revenue")
ax.set_xlabel("Group (numbers are arbitrary)")
ax.legend(["Share of customers", "Share of revenue"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Your answer.** Which group matters most for revenue compared with its size, and which matters least?

## Step 4. Check against JMP

JMP is the ground truth here. First export the buyers table. The file goes into an `exports` folder next to this notebook. It has a numeric `CustomerNumber` column you can use as the row label in JMP.

In [ ]:
Path("exports").mkdir(exist_ok=True)
export = buyers.assign(CustomerNumber=np.arange(1, len(buyers) + 1))
export = export[["CustomerNumber", "CustomerID"] + FEATURES + ["TotalSpend"]]
export.to_csv("exports/marlow_finch_customer_features.csv", index=False)
print("Saved exports/marlow_finch_customer_features.csv with", len(export), "rows")

In JMP, run K Means clustering on the four feature columns with k = 4, using the settings in the Activity Workbook, Clustering in JMP. Write down the number of customers JMP puts in each group, largest first, and type them into the cell below.

Group sizes will differ slightly between JMP and Python, and the labels will not match, which is expected. But the **pattern** should be recognisable: similar sizes and similar kinds of group. If your Python groups and your JMP groups look nothing alike, find out why before you trust either.

In [ ]:
jmp_sizes = []          # type the group sizes JMP gives you, largest first, for example [120, 90, 60, 40]

python_sizes = sorted(buyers["Group"].value_counts().tolist(), reverse=True)
if len(jmp_sizes) == K:
    check = pd.DataFrame({"Python": python_sizes, "JMP": jmp_sizes})
    check["Difference"] = check["Python"] - check["JMP"]
    print(check.to_string(index=False))
else:
    print("Python group sizes, largest first:", python_sizes)
    print("Add the four JMP sizes to jmp_sizes above and run this cell again.")

### A second opinion: hierarchical clustering

K-means needs k before it starts. **Hierarchical clustering** (here, Ward's method) starts with every customer alone and keeps merging the two closest groups, which gives a family tree called a **dendrogram**. If a completely different method gives you similar groups, you can trust them a little more.

The **adjusted Rand index** compares two sets of groupings: 1 means identical groups, and about 0 means no more alike than chance.

In [ ]:
ward = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(feature_matrix)

print("Ward group sizes, largest first:", sorted(pd.Series(ward).value_counts().tolist(), reverse=True))
print("k-means group sizes, largest first:", python_sizes)
print("Adjusted Rand index, k-means against Ward:", round(adjusted_rand_score(buyers["Group"], ward), 3))

## Step 5. Say it to a manager

Two sentences, in business language, to the marketing director. Name the groups you found, say which one matters most for revenue and why, and say what you would do differently for one of them.

**Your answer:**

**Ask Copilot.** Try this prompt, then check what it gives you against your profile table:

> Using the dataframe `buyers` with a column `Group`, draw a heatmap of the mean RecencyDays, Orders, AvgOrder and DiscountShare for each Group, with each column scaled from 0 to 1 so they can be compared.

Does the heatmap tell the same story as your profile table? If not, which one is wrong?

---
# Part 2. Exercise: segment business customers

## Step 1. Load and look

A supplier surveyed 100 of its business customers and asked them to score the supplier from 1 to 10 on 13 things, such as product quality, delivery speed and complaint resolution. The same file also records how satisfied each customer is, how likely they are to buy again, and how much they buy. **Do the customers' perceptions fall into groups, and do the groups matter?**

The clustering uses only the 13 perception scores. The other columns are held back so that we can test the groups afterwards.

In [ ]:
import re

firms = pd.read_excel("../data/Business Customer Data.xlsx", sheet_name="B2B Data")
PERCEPTIONS = [c for c in firms.columns if re.match(r"P\d+ ", c)]

print("firms:", firms.shape[0], "  perception scores used for clustering:", len(PERCEPTIONS))
print(firms[PERCEPTIONS].describe().loc[["mean", "std", "min", "max"]].round(2).T.to_string())

**What to look for.** All 13 scores sit on the same 1 to 10 kind of scale, but their averages and spreads still differ.

## Step 2. Run the model

Same sweep as before, over a smaller range because there are only 100 firms. The scores are first rescaled so that each one has an average of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(firms[PERCEPTIONS])      # rescale each score to average 0, standard deviation 1

rows = []
for k in range(2, 8):
    km_k = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(scaled)
    rows.append({"k": k, "inertia": km_k.inertia_, "silhouette": silhouette_score(scaled, km_k.labels_)})
sweep_b = pd.DataFrame(rows)
print(sweep_b.round({"inertia": 1, "silhouette": 3}).to_string(index=False))

**Choose k yourself.** Set `K_BUSINESS` in the cell below. We suggest 3 to start with, then try 2 and 4 and see what changes in the steps that follow.

In [ ]:
K_BUSINESS = 3

km_b = KMeans(n_clusters=K_BUSINESS, n_init=N_INIT, random_state=SEED).fit(scaled)
firms["Group"] = km_b.labels_
print("Firms in each group:", firms["Group"].value_counts().sort_index().tolist())
print("Silhouette at this k:", round(silhouette_score(scaled, km_b.labels_), 3))

## Step 3. Read the output in plain English

First, what each group thinks. The table shows each group's average score on each perception, in **standard deviations from the overall average** (0 is average, minus 1 is one standard deviation below).

In [ ]:
z = pd.DataFrame(scaled, columns=PERCEPTIONS)
z["Group"] = firms["Group"].to_numpy()
print(z.groupby("Group").mean().round(2).T.to_string())

Now the test that matters. The clustering never saw these columns. If the groups differ on them, the groups are telling us something real.

In [ ]:
outcomes = ["OR1 - Satisfaction", "OR3 - Likely to Purchase", "OR4 - Purchase Level"]
print(firms.groupby("Group")[outcomes].mean().round(2).to_string())
print()
print(pd.crosstab(firms["Group"], firms["D1 - Customer Type"]).to_string())

**What to look for.** Which perceptions mark out each group? Do satisfaction, intention to buy and purchase level differ between groups? How long have the firms in each group been customers?

**Be honest about how clear the groups are.** The silhouette from step 2 is a rough guide. As a working rule in this module, values around 0.4 and above mean clearly separated groups, and values around 0.2 or below mean groups with blurry edges: useful segments, but not natural species.

**Your answer.** Is this a clear or a blurry segmentation? What in the table makes you say so?

## Step 4. Check against JMP

In JMP, run Hierarchical clustering with Ward's method on the same 13 columns, using the settings in the Activity Workbook. Read the dendrogram, cut it at the same number of groups, and compare group sizes with the cell below. The Activity Workbook has the steps.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

tree = linkage(scaled, method="ward")
fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(tree, truncate_mode="lastp", p=20, no_labels=True, ax=ax)
ax.set_title("Dendrogram (Ward). The tall joins are the big decisions.")
plt.tight_layout()
plt.show()

ward_b = fcluster(tree, t=K_BUSINESS, criterion="maxclust")
print("Ward group sizes, largest first:  ", sorted(pd.Series(ward_b).value_counts().tolist(), reverse=True))
print("k-means group sizes, largest first:", sorted(firms["Group"].value_counts().tolist(), reverse=True))
print("Adjusted Rand index, k-means against Ward:", round(adjusted_rand_score(firms["Group"], ward_b), 3))

## Step 5. Say it to a manager

Two sentences to the supplier's sales director. Say whether the customers fall into groups, how sure you are, and which group you would visit first.

**Your answer:**

**Ask Copilot.** Try:

> For the dataframe `firms`, which three perception columns differ most between Groups? Show a bar chart of the standardised group means for those three.

Check its three columns against the standardised table above.

---
# Stretch

Quick finishers: pick one of these and use Copilot to write the code. Check every number against what you already know.

- Re-run Part 1 with the sweep extended to k = 12. Does the elbow become clearer or less clear?
- Add a `ReturnRate` column (the share of a customer's orders with `Returned` equal to 1) to the SQL query, then use it as a fifth clustering column. What happens to the silhouette and to the group sizes? Does the new column help?
- In Part 2, run the clustering on only the five perceptions that differ most between groups. Do the same groups come back?

In [ ]:
# Your space.